# Predicting F1 Pit Stops

In [1]:
import joblib
import optuna
import pandas as pd
import numpy as np


from catboost import CatBoostClassifier
from xgboost import XGBClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from IPython.display import clear_output
from pathlib import Path
import sys

cwd = Path().cwd()
project_root = cwd if (cwd / "utils").exists() else cwd.parent
sys.path.append(str(project_root))
from utils.model_hub import Models_hub

RANSOM_SEED = 42
models_hub = Models_hub(random_state=RANSOM_SEED)
save_path = "src/Predicting F1 Pit Stops/saved_models"
pre_trained_models = {
    XGBClassifier: False
}

/Users/georgiy/.pyenv/versions/3.12.8/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### EDA

In [2]:
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test.csv')

train.describe()

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
count,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000
mean,219569.500000,2023.523544,0.136118,23.105909,1.789113,14.158231,9.630339,90.948735,-3.770040,-25.721759,0.337661,0.101542,0.198982
std,126768.942943,1.024930,0.342915,16.958261,0.950194,9.801338,5.278770,19.772769,43.945759,54.766573,0.253277,4.006765,0.399235
min,0.000000,2022.000000,0.000000,1.000000,1.000000,1.000000,1.000000,67.694000,-2403.895000,-274.564000,0.012821,-18.000000,0.000000
25%,109784.750000,2023.000000,0.000000,9.000000,1.000000,6.000000,5.000000,82.621000,-8.884000,-46.566250,0.129870,-1.000000,0.000000
50%,219569.500000,2024.000000,0.000000,19.000000,2.000000,12.000000,10.000000,90.521000,-0.295000,-20.994000,0.269231,0.000000,0.000000
75%,329354.250000,2024.000000,0.000000,36.000000,2.000000,20.000000,14.000000,98.471000,0.115000,-6.199000,0.513158,2.000000,0.000000
max,439139.000000,2025.000000,1.000000,78.000000,8.000000,77.000000,20.000000,2507.607000,2423.932000,2412.026000,1.000000,18.000000,1.000000


* **id** — уникальный идентификатор записи
* **Driver** — идентификатор гонщика (например, D109)
* **Compound** — тип шин (HARD, MEDIUM, SOFT)
* **Race** — название гонки (этап чемпионата)
* **Year** — год проведения гонки
* **PitStop** — был ли пит-стоп на текущем круге (1 — да, 0 — нет)
* **LapNumber** — номер текущего круга
* **Stint** — номер отрезка на текущем комплекте шин (между пит-стопами)
* **TyreLife** — возраст шин (количество кругов, пройденных на этом комплекте)
* **Position** — текущая позиция гонщика в гонке
* **LapTime (s)** — время круга в секундах
* **LapTime_Delta** — изменение времени круга относительно предыдущего круга или базового значения
* **Cumulative_Degradation** — накопленная деградация шин (ухудшение их эффективности)
* **RaceProgress** — прогресс гонки (доля пройденной дистанции, от 0 до 1)
* **Position_Change** — изменение позиции относительно предыдущего круга
* **PitNextLap** — целевая переменная: будет ли пит-стоп на следующем круге (1 — да, 0 — нет)


In [3]:
X_merged = pd.concat([train.drop(columns=['PitStop']), test], axis=0)
X_merged['LapTime_normalized'] = X_merged.groupby('Race')['LapTime (s)'].transform(lambda x: (x - x.mean()) / x.std())

X_train, X_test, y_train, y_test = train_test_split(
    train.drop(columns=['PitStop']),
    train['PitStop'],
    test_size=0.2,
    random_state=42,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
)

categorical_features = ['Driver', 'Compound', 'Race']
numerical_features = ['Year', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)
X_train.shape, X_val.shape, X_test.shape

((263484, 891), (87828, 891), (87828, 891))

In [4]:
models_hub.fit_predict(
    LogisticRegression, 
    X_train, 
    y_train, 
    model_name="Logistic Regression", 
    params = {"random_state": RANSOM_SEED, "max_iter": 1000})
models_hub.leaderboard()

,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc,params
0,Logistic Regression,0.6175,0.0013,0.8741,0.6119,0.2226,0.3264,0.8240,"{'random_state': 42, 'max_iter': 1000}"


In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 5000),
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': RANSOM_SEED,
        'eval_metric': 'auc',
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred_proba)

if pre_trained_models.get(XGBClassifier, False):
    model_file = Path(save_path) / "XGBoost.pkl"
    if model_file.exists():
        xgb_model = joblib.load(model_file)
        print(f"Model loaded from {model_file}")
    else:
        print(f"Model file not found: {model_file}")
        xgb_model = None
else:
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50)
    
    best_params = study.best_params
    
    clear_output(wait=True)
    _, xgb_model = models_hub.fit_predict(
        model_class=XGBClassifier,
        X=X_train,
        y=y_train,
        model_name='XGBoost',
        params=best_params,
        n_splits=3,
        return_model=True
    )
    
    Path(save_path).mkdir(parents=True, exist_ok=True)
    model_file = Path(save_path) / "XGBoost.pkl"
    joblib.dump(xgb_model, model_file)
    print(f"Model saved to {model_file}")

models_hub.leaderboard()


[I 2026-05-05 09:45:19,481] A new study created in memory with name: no-name-39865b32-5870-4b4e-b77f-499f001eaf66
[I 2026-05-05 09:45:30,050] Trial 0 finished with value: 0.9072369968246669 and parameters: {'n_estimators': 4177, 'max_depth': 3, 'learning_rate': 0.12676107382760377, 'subsample': 0.5634554185733689, 'colsample_bytree': 0.5241413060376465}. Best is trial 0 with value: 0.9072369968246669.
[I 2026-05-05 09:45:32,160] Trial 1 finished with value: 0.9099037978644997 and parameters: {'n_estimators': 619, 'max_depth': 5, 'learning_rate': 0.16235881843567598, 'subsample': 0.6049929659000766, 'colsample_bytree': 0.6310022619886191}. Best is trial 1 with value: 0.9099037978644997.
[I 2026-05-05 09:45:48,148] Trial 2 finished with value: 0.9044951087260166 and parameters: {'n_estimators': 4945, 'max_depth': 5, 'learning_rate': 0.1258902223046174, 'subsample': 0.6315856654048821, 'colsample_bytree': 0.6382215553891086}. Best is trial 1 with value: 0.9099037978644997.
[I 2026-05-05 0

({'model': 'XGBoost',
  'fit_time_sec': np.float64(9.48792662533621),
  'predict_time_sec': np.float64(0.8584974580444396),
  'accuracy': np.float64(0.8999825416344067),
  'precision': np.float64(0.7128430387526841),
  'recall': np.float64(0.4523196233208697),
  'f1': np.float64(0.5534551334652891),
  'roc_auc': '0.9083 - 0.9088',
  'params': {'n_estimators': 4305,
   'max_depth': 5,
   'learning_rate': 0.026091574549324526,
   'subsample': 0.8446497519940775,
   'colsample_bytree': 0.6267722882551761}},
 XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=0.6267722882551761, device=None,
               early_stopping_rounds=None, enable_categorical=False,
               eval_metric=None, feature_types=None, feature_weights=None,
               gamma=None, grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.026091574549324526,
         